In [1]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

2.6.0+cu124
12.4


In [2]:
# Uninstall
# !pip uninstall torch-scatter torch-sparse torch-cluster torch-spline-conv pyg-lib -y

In [3]:
# Update Torch
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [4]:
# Install PyG (automatic)
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch_geometric

In [5]:
# Verify instalation
import torch
import torch_geometric
import torch_scatter

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch_scatter.__version__)
print(torch_geometric.__version__)


2.6.0+cu124
12.4
True
2.1.2+pt26cu124
2.7.0


In [6]:
from model_PyG_cla import *
from utils import *

In [7]:
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import Compose
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam

In [8]:
import torch_geometric
print(torch_geometric.__version__)

np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

2.7.0


### Parameters

In [9]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Has transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

epochs = params["epochs"]
print("Epochs:\t", epochs)

lr = params["lr"]
print("Lr:\t", lr)

Exp:		 exp108
Raw data:	 douban
Methods:	 ['t-gae']
Has transformation: False
Dimension:	 512
Groups id:	 ['douban']
Subgroups id:	 {'douban': ['1', '2']}
Cuda:	 1
Epochs:	 500
Lr:	 0.0001


### Setup

In [10]:
# Parameters models

dataset = exp
encoders = ["GCNRes", "GIN", "GINE", "GCN", "GAT", "GATE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # Change
device = torch.device(f"cuda:{cuda}" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

NUM_HIDDEN_LAYERS = 12
HIDDEN_DIM = [1024] * NUM_HIDDEN_LAYERS + [1024]
output_feature_size = dimension # 128
# lr = 0.001
# epochs = 100

In [11]:
# Create datasets (groups and subgroups)

list_train_set = []

for group_id in groups_id:
	train_set = []
	for subgroup_id in subgroups_id[group_id]:
		train_set.append("{}_{}".format(group_id, subgroup_id))
	list_train_set.append(train_set)
list_train_set

[['douban_1', 'douban_2']]

### Create Data (PyG)

In [12]:
# Only for GIN
""" transform = Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(fill_value=1.0),
	T.ToDevice(device)
]) """

# For GIN and GINE
transform = T.Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	# T.AddSelfLoops(attr="edge_attr", fill_value="mean"),
	# T.AddLaplacianEigenvectorPE(k=3, attr_name=None, is_undirected=True),
	# T.AddRandomWalkPE(walk_length=8, attr_name=None),
	T.ToDevice(device)
])

In [13]:
print("Loading training datasets")
has_edge_attributes = False
with_mean1 = True
with_mean2 = True

list_train_loader = []

if raw_data_file == "ACM_DBLP":
	group_id = groups_id[0]
	
	# Read dataset
	b = np.load("data/ACM-DBLP.npz")

	# Node matching
	test_pairs = b["test_pairs"].astype(np.int32)
	pos_pairs = b["pos_pairs"].astype(np.int32)
	df_node_alignment_truth = pd.DataFrame(np.concatenate((test_pairs, pos_pairs), axis=0))
	df_node_alignment_truth

	# Data preparation
	subgroup_id = 1
	edge_index1 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x1 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

	subgroup_id = 2
	edge_index2 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x2 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

elif raw_data_file == "Douban Online_Offline":
	group_id = groups_id[0]
	
	# Read dataset
	a1, f1, a2, f2, test_pairs = load_douban()

	# Node matching
	test_pairs = torch.tensor(np.array(test_pairs, dtype=int)) - 1
	test_pairs = test_pairs.numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs.T)
	df_node_alignment_truth

	# Create data
	edge_index1, _ = dense_to_sparse(torch.from_numpy(a1.toarray()))
	x1 = torch.from_numpy(f1.toarray()).float()

	edge_index2, _= dense_to_sparse(torch.from_numpy(a2.toarray()))
	x2 = torch.from_numpy(f2.toarray()).float()

	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "Cora1-Cora2":
	group_id = groups_id[0]
		
	# Read dataset
	loader = load_npz("data/cora.npz")
	data = loader["adj_matrix"]
	samples = data.shape[0]
	features = data.shape[1]
	values = data.data
	coo_data = data.tocoo()
	# indices = torch.LongTensor([coo_data.row, coo_data.col])
	indices = np.array([coo_data.row, coo_data.col])

	# Node matching
	N = 2708
	test_pairs = [[i, i] for i in range(N)]
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	node_features = loader["node_attr"]
	node_features = node_features.toarray()

	edge_index1 = torch.tensor(indices, dtype=torch.long)
	x1 = torch.tensor(node_features, dtype=torch.float)

	edge_index2 = torch.tensor(indices, dtype=torch.long)
	x2 = torch.tensor(node_features, dtype=torch.float)

	with_mean1 = False
	with_mean2 = True

elif raw_data_file == "flickr-lastfm":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-lastfm.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "douban":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/douban.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "flickr-myspace":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-myspace.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "ACM-DBLP":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/ACM-DBLP.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	# edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	# edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	# edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = False
	with_mean1 = True
	with_mean2 = True

elif raw_data_file == "cora":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/cora.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	# edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	# edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	# edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = False
	with_mean1 = False
	with_mean2 = True
	
# Create data
train_loader = {}

# Include edge_attr
subgroup_id = 1
data1 = Data(x=x1, edge_index=edge_index1)
if has_edge_attributes:
	data1.edge_attr = edge_attr1
else:
	data1 = add_edge_attributes(data1)
print(subgroup_id, data1)
info_data(data1, include_vectors=True)

subgroup_id = 2
data2 = Data(x=x2, edge_index=edge_index2)
if has_edge_attributes:
	data2.edge_attr = edge_attr2
else:
	data2 = add_edge_attributes(data2)
print(subgroup_id, data2)
info_data(data2, include_vectors=True)

# Normalize
normalize_node_edge_features(data1, data2, with_mean1=with_mean1, with_mean2=with_mean2) # Important

# Apply transform
subgroup_id = 1
data1 = transform(data1)
print(subgroup_id, data1)
# info(data1)

subgroup_id = 2
data2 = transform(data2)
print(subgroup_id, data2)
# info(data2)

# [MEJORA 3] Antes: gcn_norm(..., add_self_loops=False) -> edge_weight = D^-1/2 A D^-1/2 (SIN la identidad).
# El script original usa A_hat = D^-1/2 (A+I) D^-1/2 (preprocess_graph). Ahora el modelo "ORI"/GCNRes calcula
# A_hat internamente con self-loops (gcn_norm(add_self_loops=True)); GIN/GINE no usan edge_weight.
# data.edge_index queda intacto (mismas aristas y mismo orden que edge_attr).
subgroup_id = 1
train_loader[f"{group_id}_{subgroup_id}"] = data1
print(1, data1)

subgroup_id = 2
train_loader[f"{group_id}_{subgroup_id}"] = data2
print(2, data2)

info_data(data1, include_vectors=True)
info_data(data2, include_vectors=True)

list_train_loader.append(train_loader)

Loading training datasets
1 Data(x=[3906, 538], edge_index=[2, 16328], edge_attr=[16328, 2])
Validate:	 True
Num. nodes:	 3906
Num. edges:	 16328
Num. features:	 538
Has isolated:	 False
Has loops:	 False
Is directed:	 True
Is undirected:	 False
tensor([[   0,    0,    1,  ..., 3904, 3905, 3905],
        [ 728, 3714,  160,  ..., 3671, 2166, 3790]])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
tensor([[0., 1.],
        [0., 1.],
        [0., 1.],
        ...,
        [1., 0.],
        [0., 1.],
        [0., 1.]])
2 Data(x=[1118, 538], edge_index=[2, 3022], edge_attr=[3022, 2])
Validate:	 True
Num. nodes:	 1118
Num. edges:	 3022
Num. features:	 538
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[   0,    1,    1,  ..., 1117, 1117, 1117],
    

In [14]:
df_node_alignment_truth

,0,1
0,831,322
1,1033,537
2,3328,453
3,2535,795
4,3115,612
...,...,...
1113,3309,473
1114,1051,325
1115,2819,396
1116,2869,757


### Train

### Mejoras marcadas en la sección de entrenamiento

Cada mejora está señalada en el código con un comentario `# [MEJORA n]` (aplicada) o `# [POSIBLE MEJORA X]` (solo marcada).

| Marca | Dónde | Qué cambia |
|---|---|---|
| MEJORA 1 | celda de configuración (nueva) | Hiperparámetros en un solo lugar y partición de pares en validación / evaluación |
| MEJORA 2 | celda de la función de entrenamiento | Positivos sin self-loops, pérdida balanceada, selección por validación con restauración de pesos, opciones (coseno, aumentación, recorte de gradiente) |
| MEJORA 3 | celda de datos | `gcn_norm(add_self_loops=False)` no reproduce la Â = D^-1/2 (A+I) D^-1/2 del original; ahora la calcula el modelo |
| MEJORA 4 | bucle de entrenamiento | Semilla antes de crear el modelo, `TGAE` parametrizado (reemplaza `TGAE_Original`), tiempos con sincronización CUDA, liberación de memoria |
| MEJORA 5 | celdas de evaluación | Se evalúa solo sobre `EVAL_PAIRS` para no reutilizar los pares de validación |
| MEJORA 6 | celda nueva al final de la evaluación | Compara recuperación L2, coseno y CSLS sin reentrenar |
| POSIBLE MEJORA A, B, C | celdas de similitud, embeddings y comentario final | Marcadas, no aplicadas |

**Advertencia de protocolo:** seleccionar el modelo con pares de anclaje de validación introduce supervisión, mientras que T-GAE es no supervisado. Con `USE_VALIDATION = False` se recupera el comportamiento anterior (early stopping por pérdida y evaluación sobre todos los pares). Las cifras con validación activa no son comparables con las del paper.


In [15]:
# [MEJORA 2] La función de entrenamiento se importa de fit_TGAE_subgraph.py (mismo directorio que model_PyG.py).
# Cambios frente a la versión inline:
#   - positivos = aristas originales SIN self-loops (data.pos_edge_index), no data.edge_index
#   - pérdida balanceada 0.5*(BCE pos + BCE neg) y neg_ratio configurable
#   - selección de modelo opcional por MRR en pares de validación + restauración de los mejores pesos
#   - weight_decay como parámetro, recorte de gradiente opcional, decoder coseno opcional,
#     aumentación opcional por eliminación de aristas (aug_p)
#   - sin random.sample (no tenía efecto: se suman las pérdidas de todos los grafos antes de backward)
# from fit_TGAE_subgraph import (fit_TGAE_subgraph, evaluate_alignment, evaluate_alignment_mean, make_val_split, set_seed)


In [16]:
"""Entrenamiento T-GAE para subgraph alignment (PyTorch Geometric).

Requiere el modelo refactorizado de model_PyG.py (edge_weight / edge_attr por nombre).

Cambios respecto a la versión anterior de fit_TGAE_subgraph:
  1. Los positivos de la reconstrucción son las aristas ORIGINALES sin self-loops
	 (data.pos_edge_index), no data.edge_index (que trae self-loops de AddSelfLoops).
  2. Selección de modelo opcional por alineamiento en un conjunto de validación
	 (val_pairs) y restauración de los mejores pesos. Sin val_pairs: early stopping
	 por pérdida, como antes.
  3. Decoder opcional "cosine" (coseno / tau + sesgo aprendible) además del producto interno.
  4. Aumentación opcional por eliminación de aristas (aug_p > 0): se codifica el grafo
	 perturbado y se reconstruye el grafo original (Ec. 7 del paper, aplicada por grafo).
  5. Pérdida balanceada: 0.5 * (BCE positivos + BCE negativos), con neg_ratio configurable.
  6. Opcional: recorte de gradiente, AdamW, semilla.
  7. Métricas de alineamiento (Hit@K, MRR) con similitud coseno, L2 o CSLS.
"""
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling, remove_self_loops
from tqdm import tqdm


GCNRES_NAMES = ("GCNRes", "ORI")  # "ORI" es el nombre que usa el notebook para la capa original

# ----------------------------------------------------------------------------
# Utilidades
# ----------------------------------------------------------------------------
def set_seed(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)


def make_val_split(pairs, val_frac=0.2, seed=0):
	"""Divide pares de anclaje [K, 2] en (validación, evaluación).

	Reporta las métricas finales SOLO sobre el segundo conjunto.
	"""
	pairs = np.asarray(pairs)
	idx = np.random.default_rng(seed).permutation(len(pairs))
	n_val = int(round(val_frac * len(pairs)))
	return pairs[idx[:n_val]], pairs[idx[n_val:]]


def prepare_graph(data):
	"""Guarda las aristas originales sin self-loops (objetivo de reconstrucción)."""
	if getattr(data, "pos_edge_index", None) is None:
		data.pos_edge_index, _ = remove_self_loops(data.edge_index)
	return data


def drop_undirected_edges(edge_index, edge_attr, p, num_nodes):
	"""Elimina cada arista no dirigida con probabilidad p (ambos sentidos a la vez).

	Conserva los self-loops y mantiene edge_attr alineado con edge_index.
	"""
	row, col = edge_index
	key = torch.minimum(row, col) * num_nodes + torch.maximum(row, col)
	uniq, inv = torch.unique(key, return_inverse=True)
	keep_undirected = torch.rand(uniq.numel(), device=edge_index.device) >= p
	keep = keep_undirected[inv] | (row == col)
	edge_index = edge_index[:, keep]
	if edge_attr is not None:
		edge_attr = edge_attr[keep]
	return edge_index, edge_attr


def _encode(model, data, encoder, aug_p=0.0):
	"""Llama al modelo según el encoder. GCNRes recibe aristas sin loops y el modelo
	calcula Â = D^-1/2 (A+I) D^-1/2 internamente; GCN recibe pesos crudos (ninguno)."""
	ei = data.pos_edge_index if encoder in GCNRES_NAMES else data.edge_index
	ea = data.edge_attr if encoder in ("GINE", "GATE") else None
	if aug_p > 0:
		ei, ea = drop_undirected_edges(ei, ea, aug_p, data.num_nodes)
	kw = {"edge_attr": ea} if ea is not None else {}
	return model(data.x, ei, **kw)


def _edge_logits(z, pos, neg, decoder, tau, bias):
	if decoder == "cosine":
		z = F.normalize(z, dim=1)
	pos_logits = (z[pos[0]] * z[pos[1]]).sum(dim=1)
	neg_logits = (z[neg[0]] * z[neg[1]]).sum(dim=1)
	if decoder == "cosine":
		pos_logits = pos_logits / tau + bias
		neg_logits = neg_logits / tau + bias
	return pos_logits, neg_logits


# ----------------------------------------------------------------------------
# Evaluación de alineamiento
# ----------------------------------------------------------------------------
def _mean_topk(a, b, k, chunk=2048):
	"""Para cada fila de a: media de las k mayores similitudes coseno con las filas de b."""
	out = []
	for i in range(0, a.size(0), chunk):
		out.append((a[i:i + chunk] @ b.T).topk(k, dim=1).values.mean(dim=1))
	return torch.cat(out)


@torch.no_grad()
def evaluate_alignment(z1, z2, pairs, retrieval="cosine", csls_k=None, ks=(1, 5, 10, 50)):
	"""Hit@K y MRR de z1 (grafo 1) hacia z2 (grafo 2). pairs: [K, 2] (idx en 1, idx en 2)."""
	pairs = torch.as_tensor(np.asarray(pairs), dtype=torch.long, device=z1.device)
	src, tgt = pairs[:, 0], pairs[:, 1]
	if retrieval == "cosine":
		a, b = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
		sim = a[src] @ b.T
		if csls_k:
			r_s = _mean_topk(a[src], b, csls_k)
			r_t = _mean_topk(b, a, csls_k)
			sim = 2 * sim - r_s.unsqueeze(1) - r_t.unsqueeze(0)
	elif retrieval == "l2":
		if csls_k:
			raise ValueError("csls_k solo está soportado con retrieval='cosine'")
		sim = -torch.cdist(z1[src], z2)
	else:
		raise ValueError("retrieval debe ser 'cosine' o 'l2'")
	true = sim[torch.arange(len(src), device=sim.device), tgt].unsqueeze(1)
	rank = (sim > true).sum(dim=1) + 1
	out = {f"hit@{k}": (rank <= k).float().mean().item() for k in ks}
	out["mrr"] = (1.0 / rank.float()).mean().item()
	return out


def evaluate_alignment_mean(z1, z2, pairs, **kw):
	"""Promedio de grafo1->grafo2 y grafo2->grafo1 (equivale a mode="mean" de PlanetAlign)."""
	pairs = np.asarray(pairs)
	a = evaluate_alignment(z1, z2, pairs, **kw)
	b = evaluate_alignment(z2, z1, pairs[:, ::-1].copy(), **kw)
	return {k: (a[k] + b[k]) / 2 for k in a}


# ----------------------------------------------------------------------------
# Entrenamiento
# ----------------------------------------------------------------------------
def fit_TGAE_subgraph(model, encoder, exp, group_id, subgroups_id, train_loader, train_set,
					no_samples, epochs, lr, weight_decay,
					val_pairs=None, eval_every=5, select_metric="mrr",
					retrieval="cosine", csls_k=None,
					decoder="dot", tau=0.2, neg_ratio=1.0,
					aug_p=0.0, grad_clip=None, optimizer_name="adam",
					patience=50, delta=1e-4, warmup=10, seed=None, save_outputs=True):
	"""Entrena el encoder y guarda embeddings y pérdida (mismas rutas que antes).

	no_samples se conserva por compatibilidad de firma; la pérdida se promedia
	sobre len(train_set). Devuelve un dict con el resumen del entrenamiento.
	"""
	if seed is not None:
		set_seed(seed)
	dev = next(model.parameters()).device
	if val_pairs is not None and len(subgroups_id) < 2:
		raise ValueError("La validación requiere dos subgrafos en subgroups_id")

	for ts in train_set:
		prepare_graph(train_loader[ts])
	for sg in subgroups_id:
		prepare_graph(train_loader[f"{group_id}_{sg}"])

	# Sesgo aprendible solo para el decoder coseno (sin él, la pérdida satura).
	bias = torch.zeros((), device=dev, requires_grad=True) if decoder == "cosine" else None
	param_groups = [{"params": list(model.parameters())}]
	if bias is not None:
		param_groups.append({"params": [bias], "weight_decay": 0.0})
	opt_cls = torch.optim.AdamW if optimizer_name == "adamw" else torch.optim.Adam
	optimizer = opt_cls(param_groups, lr=lr, weight_decay=weight_decay)

	use_val = val_pairs is not None
	list_loss, history = [], []
	best_score, best_epoch, best_state = -float("inf"), 0, None
	best_loss, bad_count, stopped_epoch = None, 0, None
	max_bad = max(1, patience // eval_every) if use_val else patience

	model.train()
	loop_obj = tqdm(range(1, epochs + 1))
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		optimizer.zero_grad()
		loss = torch.zeros((), device=dev)

		for ts in train_set:
			data = train_loader[ts]
			z = _encode(model, data, encoder, aug_p)

			pos = data.pos_edge_index  # objetivo: grafo original, sin self-loops
			neg = negative_sampling(
				edge_index=pos, num_nodes=z.size(0),
				num_neg_samples=int(neg_ratio * pos.size(1)), method="sparse",
			)
			pos_logits, neg_logits = _edge_logits(z, pos, neg, decoder, tau, bias)
			loss = loss + 0.5 * (
				F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
				+ F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
			)

		loss = loss / len(train_set)
		loss.backward()
		if grad_clip is not None:
			torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
		optimizer.step()

		loss_value = loss.item()
		list_loss.append(loss_value)
		loop_obj.set_postfix_str(f"Loss: {loss_value:.4f}")

		if use_val:
			if epoch % eval_every == 0 or epoch == epochs:
				model.eval()
				with torch.no_grad():
					z1 = _encode(model, train_loader[f"{group_id}_{subgroups_id[0]}"], encoder)
					z2 = _encode(model, train_loader[f"{group_id}_{subgroups_id[1]}"], encoder)
					metrics = evaluate_alignment(z1, z2, val_pairs, retrieval, csls_k)
				model.train()
				metrics["epoch"] = epoch
				history.append(metrics)
				score = metrics[select_metric]
				loop_obj.set_postfix_str(f"Loss: {loss_value:.4f} val_{select_metric}: {score:.4f}")
				if score > best_score + 1e-6:
					best_score, best_epoch, bad_count = score, epoch, 0
					# En CPU para no duplicar memoria de GPU.
					best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
				else:
					bad_count += 1
				if epoch >= warmup and bad_count >= max_bad:
					stopped_epoch = epoch
					print(f"Early stopping (validación) en la época {epoch}; mejor época: {best_epoch}")
					break
		else:
			if epoch >= warmup:
				if best_loss is None or loss_value < best_loss - delta:
					best_loss, bad_count = loss_value, 0
				else:
					bad_count += 1
					if bad_count >= max_bad:
						stopped_epoch = epoch
						print(f"Early stopping (pérdida) en la época {epoch}")
						break

	if use_val and best_state is not None:
		model.load_state_dict(best_state)
		print(f"Pesos restaurados de la época {best_epoch} (val_{select_metric}={best_score:.4f})")

	# Embeddings finales
	model.eval()
	list_df_node_embeddings = []
	with torch.no_grad():
		for subgroup_id in subgroups_id:
			data = train_loader[f"{group_id}_{subgroup_id}"]
			z = _encode(model, data, encoder)
			df = pd.DataFrame(z.cpu().numpy())
			df["subgroup_id"] = [subgroup_id] * len(df)
			df.insert(0, "Id", range(len(df)))  # solo para datos que no son MS
			list_df_node_embeddings.append(df)
	df_node_embeddings_concat = pd.concat(list_df_node_embeddings, ignore_index=True)

	if save_outputs:
		emb_dir = f"experiments/output/{exp}/node_embeddings"
		loss_dir = f"experiments/output/{exp}/loss"
		os.makedirs(emb_dir, exist_ok=True)
		os.makedirs(loss_dir, exist_ok=True)
		df_node_embeddings_concat.to_csv(f"{emb_dir}/{encoder}_{group_id}.csv", index=False)
		np.save(f"{loss_dir}/{encoder}_{group_id}.npy", np.array(list_loss))

	if torch.cuda.is_available():
		torch.cuda.empty_cache()

	return {
		"best_epoch": best_epoch if use_val else None,
		"best_val_score": best_score if use_val else None,
		"stopped_epoch": stopped_epoch,
		"epochs_run": len(list_loss),
		"loss": list_loss,
		"val_history": history,
	}

In [17]:
# [MEJORA 1] Hiperparámetros de entrenamiento y partición de pares en un solo lugar
SEED = 0
WEIGHT_DECAY = 5e-4          # antes fijo dentro de la función
HEADS = 4                    # solo GAT / GATE

# Selección de modelo (usa anclajes -> supervisión; poner False para el protocolo no supervisado)
USE_VALIDATION = False
VAL_FRAC = 0.2
EVAL_EVERY = 5
SELECT_METRIC = "mrr"        # "mrr" | "hit@1" | "hit@5" | "hit@10" | "hit@50"
RETRIEVAL = "l2"             # "l2" (como el paper y la evaluación final) | "cosine"

# Objetivo de entrenamiento (valores por defecto = comportamiento previo / paper)
DECODER = "dot"              # "dot" | "cosine" (con TAU y sesgo aprendible)
TAU = 0.2
NEG_RATIO = 1.0              # negativos por positivo
AUG_P = 0.0                  # probabilidad de eliminar cada arista al codificar (experimento A/B)
GRAD_CLIP = None             # p. ej. 1.0

# Con validación el criterio se mide en épocas: 50 = 10 evaluaciones sin mejora con EVAL_EVERY = 5
PATIENCE = 50 if USE_VALIDATION else 10
WARMUP = 10

# Nombre en `encoders` -> tipo de convolución de model_PyG.TGAE
ENCODER_TO_CONV = {"GCNRes": "gcnres", "GIN": "gin", "GINE": "gine",
				   "GCN": "gcn", "GAT": "gatv2", "GATE": "gatv2e"}

all_pairs = df_node_alignment_truth.values.astype(np.int64)   # [K, 2]: (nodo en grafo 1, nodo en grafo 2)
if USE_VALIDATION:
	VAL_PAIRS, EVAL_PAIRS = make_val_split(all_pairs, val_frac=VAL_FRAC, seed=SEED)
else:
	VAL_PAIRS, EVAL_PAIRS = None, all_pairs
df_eval_truth = pd.DataFrame(EVAL_PAIRS)
print("Pares totales:", len(all_pairs), "| validación:", 0 if VAL_PAIRS is None else len(VAL_PAIRS),
	  "| evaluación:", len(EVAL_PAIRS))


Pares totales: 1118 | validación: 0 | evaluación: 1118


In [18]:
encoders

['GCNRes', 'GIN', 'GINE', 'GCN', 'GAT', 'GATE']

In [19]:
list_train_set

[['douban_1', 'douban_2']]

In [20]:
list_train_loader

[{'douban_1': Data(x=[3906, 538], edge_index=[2, 16328], edge_attr=[16328, 2]),
  'douban_2': Data(x=[1118, 538], edge_index=[2, 3022], edge_attr=[3022, 2])}]

In [21]:
# [MEJORA 4] Un solo modelo parametrizado (model_PyG.TGAE) en lugar de una clase por encoder
list_runtime = []
list_fit_info = []

for encoder in encoders:
	for i, train_loader in enumerate(list_train_loader):
		group_id = groups_id[i]
		subgroups_id_ = subgroups_id[group_id]
		train_set = list_train_set[i]

		no_samples = len(train_set)
		input_dim = train_loader[train_set[0]].num_node_features
		edge_dim = train_loader[train_set[0]].edge_attr.size(1) if encoder in ("GINE", "GATE") else None

		# [MEJORA 4a] Semilla ANTES de construir el modelo: antes cada encoder continuaba el flujo aleatorio
		# del anterior (resultados dependientes del orden de `encoders`) y los pesos iniciales no se repetían.
		set_seed(SEED)
		model = TGAE(ENCODER_TO_CONV[encoder], NUM_HIDDEN_LAYERS, input_dim, HIDDEN_DIM,
					 output_feature_size, edge_dim=edge_dim, heads=HEADS).to(device)

		print("Fitting model")
		print(encoder, dataset, train_set, lr, epochs, input_dim, output_feature_size, no_samples)

		# [MEJORA 4b] CUDA es asíncrono: sincronizar antes y después para medir el tiempo real.
		# Nota: con validación activa, el tiempo incluye las evaluaciones periódicas.
		if device.type == "cuda":
			torch.cuda.synchronize(device)
		start_time = time.perf_counter()

		info_fit = fit_TGAE_subgraph(
			model, encoder, dataset, group_id, subgroups_id_, train_loader, train_set, no_samples,
			epochs, lr, WEIGHT_DECAY,
			val_pairs=VAL_PAIRS, eval_every=EVAL_EVERY, select_metric=SELECT_METRIC, retrieval=RETRIEVAL,
			decoder=DECODER, tau=TAU, neg_ratio=NEG_RATIO, aug_p=AUG_P, grad_clip=GRAD_CLIP,
			patience=PATIENCE, warmup=WARMUP,
		)

		if device.type == "cuda":
			torch.cuda.synchronize(device)
		list_runtime.append(time.perf_counter() - start_time)
		list_fit_info.append({"encoder": encoder, "group_id": group_id,
							  **{k: v for k, v in info_fit.items() if k not in ("loss", "val_history")}})

		# [MEJORA 4c] Liberar el modelo (12 capas x 1024) antes del siguiente encoder
		del model
		if device.type == "cuda":
			torch.cuda.empty_cache()

df_fit = pd.DataFrame(list_fit_info)
df_fit

Fitting model
GCNRes exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 132:  26%|██▌       | 131/500 [00:13<00:37,  9.87it/s, Loss: 0.4651]


Early stopping (pérdida) en la época 132
Fitting model
GIN exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 62:  12%|█▏        | 61/500 [00:15<01:52,  3.90it/s, Loss: 0.4352]


Early stopping (pérdida) en la época 62
Fitting model
GINE exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


...

Epoch: 86:  17%|█▋        | 85/500 [00:15<01:15,  5.46it/s, Loss: 0.4143]


Early stopping (pérdida) en la época 86
Fitting model
GATE exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 141:  28%|██▊       | 140/500 [00:27<01:09,  5.16it/s, Loss: 0.4022]


Early stopping (pérdida) en la época 141


,encoder,group_id,best_epoch,best_val_score,stopped_epoch,epochs_run
0,GCNRes,douban,None,None,132,132
1,GIN,douban,None,None,62,62
2,GINE,douban,None,None,149,149
3,GCN,douban,None,None,27,27
4,GAT,douban,None,None,86,86
5,GATE,douban,None,None,141,141


### Similarity analysis

In [22]:
# [POSIBLE MEJORA A - no aplicada] k=1 con NearestNeighbors asigna el vecino más cercano de CADA nodo del grafo 1 de
# forma independiente (no biyectiva) y evalúa todos los nodos aunque solo los anclajes tengan verdad conocida.
# Alternativas: asignación uno a uno (hungarian/greedy_hungarian de utils.py) o evaluate_alignment sobre EVAL_PAIRS.
k = 1 # Change
# list_count_meta = []

for encoder in encoders:
	for group_id in groups_id:
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
		# Id, 0, 1,	2, ..., subgroup_id

		subgroups_id_ = subgroups_id[group_id]

		# Calculate distance matrix (KNN)
		knn = NearestNeighbors(n_neighbors=k, metric="euclidean")

		df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
		x = df_node_embeddings.iloc[:, 1:-1].values

		df_node_alignment = pd.DataFrame()
		df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

		for subgroup_id in subgroups_id_[1:]:
			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
			y = df_node_embeddings.iloc[:, 1:-1].values

			knn.fit(y)
			distances, indices = knn.kneighbors(x)
			indices = indices.squeeze() # (N,)

			df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
			
			""" df_node_alignment[f"distances"] = distances
			avg_distances = df_node_alignment["distances"].mean()
			df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
			df_node_alignment """
		
		# Save previus node alignment
		df_node_alignment.to_csv(f"experiments/output/{exp}/filter_raw/node_alignment_{encoder}_{group_id}.csv")
		display(df_node_alignment)

		# Find node alignment for all datasets
		""" df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
		list_count_meta.append([encoder, group_id, len(df_node_alignment_filter), len(df_node_alignment)])
		# print(len(df_node_alignment_filter))
		# display(df_node_alignment_filter)
		print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")
		
		# Save common node id
		# common_node_id = df_node_alignment_filter.iloc[:, 0].values
		# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id)) """

		# Save common node id as .csv
		df_common_node_id = df_node_alignment.iloc[:, [0, 1]].copy() # df_common_node_id = df_node_alignment_filter.iloc[:, [0, 1]].copy()
		df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
		df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

# Intersection and Union
""" list_common_node_id = []
for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		list_common_node_id.append(common_node_id)

	common_node_id_union = list(set.union(*map(set, list_common_node_id)))
	common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))

	# Save common node id
	np.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))
	np.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) """


,douban_1,douban_2
0,0,787
1,1,881
2,2,973
3,3,892
4,4,634
...,...,...
3901,3901,248
3902,3902,314
3903,3903,431
3904,3904,826


,douban_1,douban_2
0,0,1066
1,1,111
2,2,35
3,3,938
4,4,634
...,...,...
3901,3901,1094
3902,3902,314
3903,3903,1054
3904,3904,190


,douban_1,douban_2
0,0,853
1,1,366
2,2,405
3,3,938
4,4,559
...,...,...
3901,3901,406
3902,3902,314
3903,3903,315
3904,3904,948


,douban_1,douban_2
0,0,1066
1,1,1020
2,2,918
3,3,892
4,4,634
...,...,...
3901,3901,749
3902,3902,314
3903,3903,937
3904,3904,734


,douban_1,douban_2
0,0,1066
1,1,622
2,2,630
3,3,793
4,4,634
...,...,...
3901,3901,277
3902,3902,314
3903,3903,506
3904,3904,842


,douban_1,douban_2
0,0,1066
1,1,1084
2,2,837
3,3,793
4,4,634
...,...,...
3901,3901,30
3902,3902,314
3903,3903,277
3904,3904,734


' list_common_node_id = []\nfor encoder in encoders:\n\tfor group_id in groups_id:\n\t\t# Read common node\n\t\tcommon_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")\n\t\tlist_common_node_id.append(common_node_id)\n\n\tcommon_node_id_union = list(set.union(*map(set, list_common_node_id)))\n\tcommon_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))\n\n\t# Save common node id\n\tnp.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))\n\tnp.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) '

### Evaluation

In [23]:
df_node_alignment_truth

,0,1
0,831,322
1,1033,537
2,3328,453
3,2535,795
4,3115,612
...,...,...
1113,3309,473
1114,1051,325
1115,2819,396
1116,2869,757


In [24]:
list_accuracy = []
list_encoder = []

for encoder in encoders:
	for group_id in groups_id:
		# Read node alignment
		df_common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
		# df_common_node_id[[0, 1]] = np.sort(df_common_node_id[[0, 1]], axis=1)

		# Evaluation
		# [MEJORA 5] Evaluar solo sobre EVAL_PAIRS (los pares de validación no se reutilizan)
		df_matching = df_common_node_id.merge(df_eval_truth, on=[0, 1], how="inner")

		# Accuracy
		accuracy = len(df_matching) / len(df_eval_truth)
		
		list_accuracy.append(accuracy)
		list_encoder.append(encoder)

##### Summary

In [25]:
dict_summary = {
	"Dataset": [raw_data_file] * len(encoders),
	"Model": list_encoder,
	"Accuracy": list_accuracy,
	"Runtime": list_runtime
}
df_summary = pd.DataFrame(dict_summary)
df_summary

,Dataset,Model,Accuracy,Runtime
0,douban,GCNRes,0.284436,14.461635
1,douban,GIN,0.415921,16.843277
2,douban,GINE,0.288014,42.804452
3,douban,GCN,0.214669,3.627169
4,douban,GAT,0.293381,16.744202
5,douban,GATE,0.305009,28.308177


### Evaluation (based on PlanetAlign)

In [26]:
# [POSIBLE MEJORA B - no aplicada] Los embeddings (N x 1024) viajan por CSV: lento y pesado en disco.
# fit_TGAE_subgraph podría devolver z1/z2 en memoria (y guardarlos en .npy) para evitar esta lectura.
# Get node embeddings
dict_node_embeddings = {}
for encoder in encoders:
	list_node_embeddings = []
	for group_id in groups_id:
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
		# Id, 0, 1,	2, ..., subgroup_id

		for subgroup_id in subgroups_id[group_id]:
			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
			list_node_embeddings.append(df_node_embeddings)
	dict_node_embeddings[encoder] = list_node_embeddings

In [27]:
dict_node_embeddings

{'GCNRes': [        Id         0         1         2         3         4         5  \
  0        0 -0.015374  0.066439  0.036427 -0.002777 -0.028167  0.078217   
  1        1  0.014532 -0.016575 -0.102013  0.015075  0.013815 -0.110370   
  2        2  0.021064  0.011008 -0.039657 -0.057598 -0.043956  0.011835   
  3        3 -0.001358 -0.059174 -0.009650 -0.005456  0.013695 -0.105803   
  4        4  0.003719 -0.040082  0.050614 -0.086873 -0.102353  0.085079   
  ...    ...       ...       ...       ...       ...       ...       ...   
  3901  3901 -0.091143  0.575543  0.210939  0.525259 -0.123624 -0.215748   
  3902  3902  0.125469 -0.679718  0.182359 -0.360220 -0.331332  0.140029   
  3903  3903  0.037275  0.072538  0.054143  0.014076  0.006766 -0.001945   
  3904  3904  0.041958  0.167488 -0.029866 -0.003328 -0.100196  0.118639   
  3905  3905  0.028318 -0.080544  0.117172  0.060930 -0.042085 -0.046544   
  
               6         7         8  ...       503       504       505  \


In [28]:
# Evaluation
list_metrics = []

# [MEJORA 5] Evaluar solo sobre EVAL_PAIRS (si USE_VALIDATION=False son todos los pares, como antes)
df_node_alignment_truth_ = torch.tensor(EVAL_PAIRS)

for encoder in encoders:
	list_node_embeddings = dict_node_embeddings[encoder]
	# Id, 0, 1,	2, ..., subgroup_id

	emb1 = list_node_embeddings[0].iloc[:, 1:-1]
	emb2 = list_node_embeddings[1].iloc[:, 1:-1]

	emb1_ = torch.tensor(emb1.values)
	emb2_ = torch.tensor(emb2.values)

	# Similarity matrix
	# Option 1
	# S = pairwise_cosine_similarity(emb1_, emb2_)

	# Option 2
	""" S = cosine_similarity(emb1, emb2)
	S = torch.tensor(S) """

	# Option 3
	S = pairwise_distances(emb1, emb2, metric="euclidean")
	S = torch.tensor(S)
	S = -S # convert distances matrix to similarity matrix
	
	# Option 4
	""" S = torch.cdist(emb1_, emb2_, 2)
	S = -S # convert distances matrix to similarity matrix """

	hits = hits_ks_scores(S, df_node_alignment_truth_, mode="mean")
	mrr = mrr_score(S, df_node_alignment_truth_, mode="mean")
	print(encoder, hits, mrr)
	
	list_metrics.append(list(hits.values()) + [mrr])

GCNRes {1: 0.25313058495521545, 10: 0.53220035135746, 30: 0.6900715529918671, 50: 0.7517889142036438} 0.3464001715183258
GIN {1: 0.3407871276140213, 10: 0.5983899831771851, 30: 0.7312164604663849, 50: 0.7799642086029053} 0.4293224960565567
GINE {1: 0.22719141840934753, 10: 0.46556350588798523, 30: 0.5930232554674149, 50: 0.6623434722423553} 0.30604538321495056
GCN {1: 0.20483005046844482, 10: 0.4718247056007385, 30: 0.6220930218696594, 50: 0.6882826685905457} 0.2960786819458008
GAT {1: 0.22987478226423264, 10: 0.4767441749572754, 30: 0.6015205681324005, 50: 0.6498211026191711} 0.30990248918533325
GATE {1: 0.22942755371332169, 10: 0.4642218202352524, 30: 0.5854203999042511, 50: 0.6435599029064178} 0.3092137649655342


In [29]:
list_summary = []
for k in range(len(list_metrics)):
	list_summary.append([raw_data_file] + [encoders[k]] + list_metrics[k] + [list_runtime[k]])
list_summary

[['douban',
  'GCNRes',
  0.25313058495521545,
  0.53220035135746,
  0.6900715529918671,
  0.7517889142036438,
  0.3464001715183258,
  14.46163473406341],
 ['douban',
  'GIN',
  0.3407871276140213,
  0.5983899831771851,
  0.7312164604663849,
  0.7799642086029053,
  0.4293224960565567,
  16.843276533996686],
 ['douban',
  'GINE',
  0.22719141840934753,
  0.46556350588798523,
  0.5930232554674149,
  0.6623434722423553,
  0.30604538321495056,
  42.80445165291894],
 ['douban',
  'GCN',
  0.20483005046844482,
  0.4718247056007385,
  0.6220930218696594,
  0.6882826685905457,
  0.2960786819458008,
  3.6271693179151043],
 ['douban',
  'GAT',
  0.22987478226423264,
  0.4767441749572754,
  0.6015205681324005,
  0.6498211026191711,
  0.30990248918533325,
  16.74420213198755],
 ['douban',
  'GATE',
  0.22942755371332169,
  0.4642218202352524,
  0.5854203999042511,
  0.6435599029064178,
  0.3092137649655342,
  28.308177384082228]]

In [30]:
columns = ["Dataset", "Encoder"] + [f"Hits@{k}" for k in list(hits.keys())] +  ["MRR", "Runtime"]
df_summary = pd.DataFrame(list_summary, columns=columns)
df_summary

,Dataset,Encoder,Hits@1,Hits@10,Hits@30,Hits@50,MRR,Runtime
0,douban,GCNRes,0.253131,0.532200,0.690072,0.751789,0.346400,14.461635
1,douban,GIN,0.340787,0.598390,0.731216,0.779964,0.429322,16.843277
2,douban,GINE,0.227191,0.465564,0.593023,0.662343,0.306045,42.804452
3,douban,GCN,0.204830,0.471825,0.622093,0.688283,0.296079,3.627169
4,douban,GAT,0.229875,0.476744,0.601521,0.649821,0.309902,16.744202
5,douban,GATE,0.229428,0.464222,0.585420,0.643560,0.309214,28.308177


##### Comparación de métodos de recuperación (sin reentrenar)

In [31]:
df_node_alignment_truth

,0,1
0,831,322
1,1033,537
2,3328,453
3,2535,795
4,3115,612
...,...,...
1113,3309,473
1114,1051,325
1115,2819,396
1116,2869,757


In [32]:
print(len(EVAL_PAIRS))
EVAL_PAIRS

1118


array([[ 831,  322],
       [1033,  537],
       [3328,  453],
       ...,
       [2819,  396],
       [2869,  757],
       [3342,    3]])

In [33]:
# [MEJORA 6] Misma evaluación (promedio ida/vuelta, como mode="mean") con tres formas de recuperar vecinos.
# CSLS corrige el efecto "hub" (nodos que son vecino más cercano de muchos otros).
KS = (1, 5, 10, 50)
rows = []
for encoder in encoders:
	emb1, emb2 = dict_node_embeddings[encoder][0], dict_node_embeddings[encoder][1]
	# Id, 0, 1,	2, ..., subgroup_id

	z1 = torch.tensor(emb1.iloc[:, 1:-1].values, dtype=torch.float32, device=device)
	z2 = torch.tensor(emb2.iloc[:, 1:-1].values, dtype=torch.float32, device=device)
	for name, kw in {"L2": dict(retrieval="l2"),
					"Coseno": dict(retrieval="cosine"),
					"Coseno + CSLS (k=10)": dict(retrieval="cosine", csls_k=10)}.items():
		print(name, kw)
		m = evaluate_alignment_mean(z1, z2, EVAL_PAIRS, ks=KS, **kw)
		print(m)
		rows.append({"Dataset": raw_data_file, "Encoder": encoder, "Recuperación": name,
					 **{f"Hits@{k}": m[f"hit@{k}"] for k in KS}, "MRR": m["mrr"]})
	
	# ---
	S = pairwise_distances(z1.cpu().numpy(), z2.cpu().numpy(), metric="euclidean")
	S = torch.tensor(S)
	S = -S
	hits = hits_ks_scores(S, torch.tensor(EVAL_PAIRS), mode="mean")
	print(hits)
	print()
	# ---

df_retrieval = pd.DataFrame(rows)
df_retrieval

L2 {'retrieval': 'l2'}
{'hit@1': 0.2643112689256668, 'hit@5': 0.4440965950489044, 'hit@10': 0.5344364941120148, 'hit@50': 0.7517889142036438, 'mrr': 0.35556910932064056}
Coseno {'retrieval': 'cosine'}


{'hit@1': 0.2889087647199631, 'hit@5': 0.47361357510089874, 'hit@10': 0.5559033900499344, 'hit@50': 0.7544722557067871, 'mrr': 0.38000814616680145}
Coseno + CSLS (k=10) {'retrieval': 'cosine', 'csls_k': 10}
{'hit@1': 0.2906976714730263, 'hit@5': 0.48971377313137054, 'hit@10': 0.5805008858442307, 'hit@50': 0.775044709444046, 'mrr': 0.3896721452474594}
{1: 0.25313058495521545, 10: 0.53220035135746, 30: 0.6900715529918671, 50: 0.7517889142036438}

L2 {'retrieval': 'l2'}
{'hit@1': 0.36091233789920807, 'hit@5': 0.5272808521986008, 'hit@10': 0.6015205532312393, 'hit@50': 0.780411422252655, 'mrr': 0.44293341040611267}
Coseno {'retrieval': 'cosine'}
{'hit@1': 0.36180679500102997, 'hit@5': 0.5219141095876694, 'hit@10': 0.5952593684196472, 'hit@50': 0.782647579908371, 'mrr': 0.4426874816417694}
Coseno + CSLS (k=10) {'retrieval': 'cosine', 'csls_k': 10}
{'hit@1': 0.3586762100458145, 'hit@5': 0.532647579908371, 'hit@10': 0.6180679798126221, 'hit@50': 0.7996421754360199, 'mrr': 0.44785696268081665}

,Dataset,Encoder,Recuperación,Hits@1,Hits@5,Hits@10,Hits@50,MRR
0,douban,GCNRes,L2,0.264311,0.444097,0.534436,0.751789,0.355569
1,douban,GCNRes,Coseno,0.288909,0.473614,0.555903,0.754472,0.380008
2,douban,GCNRes,Coseno + CSLS (k=10),0.290698,0.489714,0.580501,0.775045,0.389672
3,douban,GIN,L2,0.360912,0.527281,0.601521,0.780411,0.442933
4,douban,GIN,Coseno,0.361807,0.521914,0.595259,0.782648,0.442687
5,douban,GIN,Coseno + CSLS (k=10),0.358676,0.532648,0.618068,0.799642,0.447857
6,douban,GINE,L2,0.239714,0.387299,0.468694,0.662791,0.314551
7,douban,GINE,Coseno,0.239714,0.374329,0.447674,0.649821,0.309521
8,douban,GINE,Coseno + CSLS (k=10),0.239714,0.396691,0.472272,0.676208,0.318345
9,douban,GCN,L2,0.225850,0.389535,0.473614,0.690519,0.310898


In [34]:
df_text = df_summary.map(lambda x: f"{x}".replace(".", ","))
df_text

,Dataset,Encoder,Hits@1,Hits@10,Hits@30,Hits@50,MRR,Runtime
0,douban,GCNRes,"0,25313058495521545","0,53220035135746","0,6900715529918671","0,7517889142036438","0,3464001715183258","14,46163473406341"
1,douban,GIN,"0,3407871276140213","0,5983899831771851","0,7312164604663849","0,7799642086029053","0,4293224960565567","16,843276533996686"
2,douban,GINE,"0,22719141840934753","0,46556350588798523","0,5930232554674149","0,6623434722423553","0,30604538321495056","42,80445165291894"
3,douban,GCN,"0,20483005046844482","0,4718247056007385","0,6220930218696594","0,6882826685905457","0,2960786819458008","3,6271693179151043"
4,douban,GAT,"0,22987478226423264","0,4767441749572754","0,6015205681324005","0,6498211026191711","0,30990248918533325","16,74420213198755"
5,douban,GATE,"0,22942755371332169","0,4642218202352524","0,5854203999042511","0,6435599029064178","0,3092137649655342","28,308177384082228"


In [35]:
# [POSIBLE MEJORA C - no aplicada] Con inicialización y negativos aleatorios una sola corrida puede engañar:
# repetir con varias semillas (SEED = 0..4) y reportar media y desviación de Hits@K y MRR.
print(NUM_HIDDEN_LAYERS)
print(HIDDEN_DIM)
print(output_feature_size)

12
[1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024]
512
